# Redis Key Schema

This notebook walks through the Redis objects used by the current synthetic identity-driven demo.

The important change is that the repo now supports multiple serving methods over the same dataset:

- legacy brute-force MAID `SINTER` retrieval
- tightened MAID `SINTER` retrieval
- direct per-MAID precomputed candidate lists with `maid_hot` scoring profiles

In [ ]:
from pathlib import Path
import json
import pandas as pd

from data.common import read_jsonl
from app.candidate import build_candidate_lookup_keys
from app.models import UserProfile

dataset_dir = Path('../data/generated/synthetic')
reports_dir = Path('../reports/generated')
metadata = json.loads((dataset_dir / 'metadata.json').read_text(encoding='utf-8'))
maids = [UserProfile.model_validate(row) for row in read_jsonl(dataset_dir / 'maids.jsonl')]
user_candidates = read_jsonl(dataset_dir / 'user_candidates.jsonl')
benchmark = json.loads((reports_dir / 'hybrid_benchmark.json').read_text(encoding='utf-8'))
metadata

## 1. Key Families

The current serving path uses these Redis key families:

- `identity:{token}` -> `maid_id`
- `maid:{maid_id}` -> full MAID profile used by `full_realtime` and the MAID `SINTER` modes
- `maid_hot:{maid_id}` -> compact scoring profile used by `precomputed_segment` and `hybrid_precompute_plus_realtime`
- `aud:{maid_id}` -> precomputed candidate campaign IDs for that MAID
- `campaign:{campaign_id}` -> static campaign metadata
- `campaign_state:{campaign_id}` -> mutable delivery state
- `fcap:{maid_id}` -> per-user delivery counters as a hash of `campaign_id -> delivery_count`
- `idx:*` -> set-based inverted indexes used only by the MAID `SINTER` modes

In [ ]:
maid = maids[0]
maid_candidate_row = user_candidates[0]
campaign_id = maid_candidate_row['candidate_ids'][0]

key_families = pd.DataFrame(
    [
        {'key': f'identity:{maid.identity_tokens[0]}', 'purpose': 'Resolve incoming token to maid_id'},
        {'key': f'maid:{maid.user_id}', 'purpose': 'Full MAID profile for realtime and SINTER modes'},
        {'key': f'maid_hot:{maid.user_id}', 'purpose': 'Compact scoring profile for precomputed modes'},
        {'key': f'aud:{maid.user_id}', 'purpose': 'Direct precomputed campaign list for this MAID'},
        {'key': f'campaign:{campaign_id}', 'purpose': 'Static campaign metadata'},
        {'key': f'campaign_state:{campaign_id}', 'purpose': 'Mutable pacing, budget, frequency cap state'},
        {'key': f'fcap:{maid.user_id}:{campaign_id}', 'purpose': 'Per-user delivery counter for that campaign'},
    ]
)
key_families

## 2. Indexed Dimensions For The MAID `SINTER` Modes

The two MAID retrieval modes still use Redis sets. They index:

- card tier
- country
- state
- device type
- device OS
- strong user segments

Campaigns that target all values on a dimension are expanded into all concrete sets during load so the request path stays simple.

In [ ]:
sample_user = maids[0]
legacy_keys = build_candidate_lookup_keys(sample_user, strong_signal_count=2, strategy='legacy_union_probe')
tightened_keys = build_candidate_lookup_keys(sample_user, strong_signal_count=2, strategy='union_probe')

pd.DataFrame({
    'legacy_union_probe': ['SINTER ' + ' '.join(group) for group in legacy_keys[:8]],
    'tightened_union_probe': ['SINTER ' + ' '.join(group) for group in tightened_keys + [''] * max(0, len(legacy_keys[:8]) - len(tightened_keys))][:8],
})

## 3. Request Flows By Method

The repo now supports five retrieval styles that share the same underlying campaign and identity data but differ in how they get from `maid_id` to candidates.

In [ ]:
flows = pd.DataFrame(
    [
        {
            'mode': 'maid_bruteforce_sinter',
            'request_flow': 'identity -> maid -> 26 SINTERs -> campaign/state + fcap hash -> filter -> rerank'
        },
        {
            'mode': 'maid_tightened_sinter',
            'request_flow': 'identity -> maid -> 3 SINTERs -> campaign/state + fcap hash -> filter -> rerank'
        },
        {
            'mode': 'precomputed_segment',
            'request_flow': 'identity -> maid_hot -> aud:{maid} -> campaign/state + fcap hash -> minimal live gating -> rerank'
        },
        {
            'mode': 'hybrid_precompute_plus_realtime',
            'request_flow': 'identity -> maid_hot -> aud:{maid} -> campaign/state + fcap hash -> live mutable gating -> rerank'
        },
        {
            'mode': 'hybrid_bitmap_gating',
            'request_flow': 'identity -> maid_hot -> aud:{maid} -> bm:servable gate -> campaign + fcap hash -> frequency check -> rerank'
        },
    ]
)
flows

## 4. Current Retrieval Summary

The benchmark artifact summarizes how much Redis work each method performs before producing validated candidates.

In [ ]:
overview = pd.DataFrame(benchmark['loadtests']).T[[
    'avg_sinter_ops',
    'avg_mode_redis_round_trips',
    'decision_path_p50_latency_ms',
    'decision_path_p99_latency_ms',
    'validated_candidate_p50_latency_ms',
    'validated_candidate_p99_latency_ms',
]].sort_index()
overview.loc[['maid_bruteforce_sinter', 'maid_tightened_sinter', 'precomputed_segment', 'hybrid_precompute_plus_realtime']]

## 5. Why Both Schemas Still Matter

The precomputed modes are the current low-latency direction, but the MAID `SINTER` paths are still useful in the demo because they show the progression:

1. a brute-force set-intersection design,
2. a tightened set-intersection design,
3. a direct precompute design that removes hot-path set algebra entirely.
4. a bitmap-gated variant that keeps precomputed candidates but reduces live campaign-state fanout.

That makes the benchmark report easier to interpret because the schema and the latency story are aligned.